# 03a · H4 pilot gate: is there any signal to train on?

Before spending 6-10 GPU-hours on the full H4 intervention, this notebook samples
chain-of-thought completions for **60 training-pool items** and answers three questions:

1. **Can the chains be parsed?** A chain that never states a letter is unusable.
2. **Will there be enough pairs?** Section 6 needs 200; below that, `k` rises to 8 once.
3. **Is the `chosen` side skill or luck?** This is the one that matters. On a 4-option task
   a base model near chance produces pairs whose chosen chain is mostly a lucky guess, and
   DPO on those teaches a preference for guessing rather than for reasoning.

Context for question 3: Llama-3.2-3B scores **0.276** on Hausa (`ha_p0`), against 0.25 chance
and a 0.318 majority-letter baseline. The prior study `dpo-reasoning-transfer` ran this same
intervention from a **68%** base on GSM8K and got **-0.73 pp**, not supported. H4 starts from
a far weaker position, so the gate exists to find out *how* weak before committing.

Nothing here is confirmatory. The completions are cached under the same key the full run uses,
so if the gate says go, none of this work is repeated.

## 0 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_DIR = "/content/hausa-med-qa"
import os
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/avvas200/hausa-med-qa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!git log -1 --format="%h %s"
# Confirm the commit above is the one you pushed, and that a GPU is attached.
# No GPU here means Runtime > Change runtime type > T4 GPU, then rerun.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # Llama-3.2 is gated; accept its license on huggingface.co first

In [ ]:
import json
import pandas as pd
from src import config as C, data as D, dpo as DPO, evaluate as E

from collections import Counter

pool = D.read_jsonl(C.DATA / "train_pool_ha.jsonl")
# Written by notebook 01b cell 17, not by notebook 01.
seg_status = json.load(open(C.DATA / "segment_status.json"))["pool"]

# Print the stem statuses before filtering. The pool marks an auto-fallback
# stem 'fallback_en_auto', not 'fallback_en', and a filter that matches only
# the latter excludes nothing while looking like it worked.
stems = Counter(seg_status[f"{r['id']}|q"] for r in pool)
print("question-stem statuses in the pool:")
for s, n in stems.most_common():
    print(f"  {s:24s} {n}")

hausa = DPO.hausa_stem_items(pool, seg_status)
pilot = DPO.pilot_subset(hausa)

n_english = sum(n for s, n in stems.items() if s.startswith("fallback_en"))
assert len(pool) - len(hausa) == n_english, (
    "the English-stem filter did not remove what the statuses say it should")
print(f"\ntraining pool      {len(pool)}")
print(f"with a Hausa stem  {len(hausa)}   ({n_english} English stems excluded, section 6)")
print(f"pilot subset       {len(pilot)}   seed {C.DPO_SEED}")
print(f"\nk={C.DPO_K} at temperature {C.DPO_TEMPERATURE} -> "
      f"{len(pilot) * C.DPO_K} chains to generate")

> **The Hausa CoT prompt has been hand-corrected** by the author and logged as
> deviation 6, before this gate was ever run — the same treatment `EVAL_PROMPTS` got
> before the baseline. Its opening sentence and its `(A, B, C ko D)` parenthetical are
> reused verbatim from the corrected eval prompt, so both address the model in the same
> register. Read it below anyway: if the parse rate comes back low, the wording is the
> first thing to suspect, not the model.
>
> Run `PYTHONPATH=. python tests/preflight_h4.py` before booking the GPU. It checks
> every non-GPU path in `src/dpo.py` against a stub model, including that a chain which
> obeys this prompt actually parses, and it takes a second.

In [ ]:
print(C.DPO_COT_PROMPT)
print("\n--- rendered for the first pilot item ---\n")
print(DPO.cot_prompt(pilot[0])[:700])

# 1 · Generate (the GPU step, resumable)

In [ ]:
CACHE = C.CACHE / f"dpo_cot_k{C.DPO_K}.jsonl"   # shared with the full run

# Generation is the only step that needs a GPU, and it is cached per item.
# Scoring depends on DPO.final_letter, so a parser fix means re-running the
# gate -- which should cost nothing. Load the model only if something is
# genuinely missing.
cached = set()
if CACHE.exists():
    with open(CACHE, encoding='utf-8') as f:
        cached = {json.loads(line)['id'] for line in f if line.strip()}
missing = [r for r in pilot if r['id'] not in cached]
print(f'{len(pilot) - len(missing)} of {len(pilot)} pilot items already cached')

if missing:
    tok, model, dtype = E.load_model(C.DPO_MODEL)
    print(C.DPO_MODEL, dtype, f'-> generating {len(missing)} item(s)')
else:
    tok = model = None
    print('nothing to generate - no model loaded, no GPU needed')

rows = DPO.generate_pool(tok, model, pilot, CACHE)
print(f'\n{len(rows)} items scored from {CACHE}')

# 2 · The gate

In [ ]:
pairs = DPO.build_pairs(rows, pilot)
report = DPO.gate_report(rows, pairs, n_full=len(hausa))

print(f"parse rate                 {report['parse_rate']:.3f}")
print(f"accuracy, all samples      {report['accuracy_all_samples']:.3f}")
print(f"accuracy, parsed only      {report['accuracy_parsed_only']:.3f}")
ref = report["reference_points"]
print(f"  chance                   {ref['chance']}")
print(f"  majority-letter baseline {ref['majority_letter_baseline_p0']}")
print(f"  greedy letter-only ha_p0 {ref['greedy_letter_only_ha_p0']}")
print(f"\nitems by number correct of {report['k']}: {report['n_correct_per_item']}")
print(f"pairs {report['pairs']}/{report['n_items']}  (yield {report['pair_yield_rate']:.3f})")
print(f"projected over the full pool: {report['projected_pairs_full_pool']} pairs "
      f"vs a threshold of {report['min_pairs_threshold']}")
print(f"\nestimated skill rate           {report['estimated_skill_rate']:.3f}")
print(f"estimated luck share of chosen {report['estimated_luck_share_of_chosen']:.3f}")

# Both locations, as notebook 02 does for baseline_results.json: Drive so it
# survives the session, and the repo so notebook 04 and the commit can see it.
for path in ("results/h4_pilot_gate.json", C.RESULTS / "h4_pilot_gate.json"):
    DPO.write_json(report, path)
print("\nwrote results/h4_pilot_gate.json (commit this one)")

# 2b · Why the chains fail

The gate says how often a chain is unusable. This says **why**, which is what the report
needs: a model that code-switches out of Hausa is a different finding from one that runs
out of tokens.

No GPU and no model weights — it re-reads the cached completions and loads only the
tokenizer, which is needed to tell a chain that was cut off at the token budget from one
that simply stopped. Safe to run in a fresh session after cells 0 and 4.

The Swahili detection is a hand-written marker list that deliberately excludes words Hausa
and Swahili share, so its count is a **lower bound** and the list wants your eye on it:
`DPO.SWAHILI_MARKERS`.

In [ ]:
from transformers import AutoTokenizer

# Everything is cached, so no model is loaded and tok=None is never called
# for generation -- the same idiom notebook 01b uses to reload translations.
CACHE = C.CACHE / f"dpo_cot_k{C.DPO_K}.jsonl"
rows = DPO.generate_pool(None, None, pilot, CACHE)
tok = AutoTokenizer.from_pretrained(C.DPO_MODEL)

fm = DPO.failure_modes(rows, tok=tok)
print(f"chains {fm['n_chains']}   parsed {fm['n_parsed']}   unparsed {fm['n_unparsed']}   parse rate {fm['parse_rate']:.3f}")
if fm['tokens']:
    t = fm["tokens"]
    print(f"tokens: median {t['median']}, at the {t['max_new']} budget {t['at_budget']}")

for name in ("unparsed_only", "all_chains"):
    g = fm[name]
    print(f"\n{name}  (n={g['n']})")
    print("  primary label")
    for lab, v in sorted(g["primary"].items(), key=lambda kv: -kv[1]["n"]):
        if v['n']:
            print(f"    {lab:14s} {v['n']:4d}  {v['prop']:.3f}")
    print("  any flag (overlapping)")
    for lab, v in sorted(g["any_of"].items(), key=lambda kv: -kv[1]["n"]):
        print(f"    {lab:14s} {v['n']:4d}  {v['prop']:.3f}")

print("\n" + fm["caveat"])

# Both locations, as the gate report does.
for path in ("results/h4_failure_modes.json",
             C.RESULTS / "h4_failure_modes.json"):
    DPO.write_json(fm, path)
print("\nwrote results/h4_failure_modes.json (commit this one)")

# 3 · Read ten chains

The numbers cannot tell you whether a chain reasons or guesses. Only reading them can.
For each: does the reasoning actually support the letter, or did it wander and land right?

In [ ]:
insp = DPO.inspection_sample(pairs, n=10)
pd.DataFrame(insp)[["id", "gold", "chosen_letter", "n_correct_of_k"]]

In [ ]:
for s in insp:
    print("=" * 78)
    print(f"{s['id']}   gold {s['gold']}   chose {s['chosen_letter']}   "
          f"{s['n_correct_of_k']} of {C.DPO_K} samples correct")
    print(s["chosen_chain"][:1200])

for path in ("results/h4_pilot_chains.csv", C.RESULTS / "h4_pilot_chains.csv"):
    pd.DataFrame(insp).to_csv(path, index=False, encoding="utf-8-sig")

# 4 · Deciding

| signal | go | stop |
|---|---|---|
| parse rate | at least 0.9 | below — read ten raw chains before concluding anything; the prompt wording is the first suspect |
| projected pairs | at least 200 | below — raise `k` to 8 once, as section 6 pre-registers |
| luck share of chosen | the lower the better | near 1.0 means the pairs are close to noise |
| the ten chains | reasoning supports the letter | chains wander and land right by accident |

The luck share is an estimate from `acc = skill + (1 - skill)/4`, not a measurement, so the
ten chains are the real evidence. If they are mostly lucky guesses, you have two honest
options, and **both are legitimate**:

- **Run H4 anyway** and report the null with the cause identified: self-generated preference
  pairs cannot bootstrap a model that is at chance in the target language. Paired with the
  prior study's null from a 68% base, that is a coherent two-study finding, and H4 stays a
  reported confirmatory hypothesis rather than a dropped one.
- **Stop here** and report H4 as not run, with this gate as the stated reason and
  `results/h4_pilot_gate.json` as the evidence.

What is *not* legitimate is switching the intervention model to MedGemma, or rewriting the
protocol, because H1-H3 results are already known. That is fitting the method to the outcome,
which is what the pre-registration exists to prevent.

One number worth carrying into the report either way: Llama's Hausa accuracy is 0.276 and the
pre-registered minimum gain is +3 pp, so **even a fully supported H4 reaches 0.306 — still
below the 0.318 majority-letter baseline.**